# Test LLM Labeling with OpenAI

This notebook mirrors `2_llm_label_groundtruth.ipynb`, but only labels a small subset of query groups for inspection.

Differences from the main notebook:

- Uses the official OpenAI SDK directly.
- Defaults to 20 query groups.
- Writes to test output files so it does not overwrite the main labeling artifacts.

## 1. Imports and Environment Loading

In [1]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Optional

import pandas as pd

In [3]:
def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


def find_repo_root(finalproject_root: Path) -> Path:
    """Return the repository root that contains Finalproject."""
    return finalproject_root.parent


def load_env_file(env_path: Path) -> None:
    """Load key=value pairs from a .env file without overriding existing environment variables."""
    if not env_path.exists():
        print(f"No .env file found at: {env_path}")
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def get_env_int(name: str, default: int) -> int:
    value = os.getenv(name)
    return default if value in (None, "") else int(value)


def get_env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    return default if value in (None, "") else float(value)


FINALPROJECT_ROOT = find_finalproject_root()
REPO_ROOT = find_repo_root(FINALPROJECT_ROOT)
ENV_PATH = REPO_ROOT / ".env"
load_env_file(ENV_PATH)

DATA_PATH = FINALPROJECT_ROOT / "data" / "all_recipes_final.csv"
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
BLINDED_ANNOTATION_ITEMS_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"

print("Finalproject root:", FINALPROJECT_ROOT)
print(".env path:", ENV_PATH)
print("Blinded annotation items:", BLINDED_ANNOTATION_ITEMS_PATH)

Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
.env path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\.env
Blinded annotation items: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\blinded_annotation_items.jsonl


## 2. Test Configuration

In [4]:
# -------------------------------------------------------
# OpenAI-compatible LiteLLM test config
# -------------------------------------------------------
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://litellm.imt-soft/v1")
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "gpt-4o-mini")
LLM_API_KEY = os.getenv("LLM_API_KEY")

# Self-hosted endpoint defaults: generous response budget and timeout.
LLM_TEMPERATURE = get_env_float("LLM_TEMPERATURE", 0.0)
LLM_MAX_TOKENS = get_env_int("LLM_MAX_TOKENS", 16384)
LLM_REQUEST_TIMEOUT_SECONDS = get_env_int("LLM_REQUEST_TIMEOUT_SECONDS", 300)

if not LLM_API_KEY:
    raise ValueError("LLM_API_KEY is required for the OpenAI-compatible LiteLLM endpoint.")

TEST_NUM_QUERY_GROUPS = 20
TEST_RESUME_LABELING = False
TEST_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "test_llm_groundtruth_labels_openai_compatible.jsonl"
TEST_QUERY_LOGS_PATH = ANNOTATION_OUTPUT_DIR / "test_llm_groundtruth_query_logs_openai_compatible.jsonl"


def normalize_openai_compatible_base_url(raw_base_url: str) -> str:
    """Normalize LiteLLM proxy URL for OpenAI-compatible SDK clients."""
    base_url = str(raw_base_url).strip().rstrip("/")
    if not base_url:
        raise ValueError("LLM_BASE_URL cannot be empty.")
    if not base_url.startswith(("http://", "https://")):
        base_url = "https://" + base_url
    if not base_url.endswith("/v1"):
        base_url = base_url + "/v1"
    return base_url


LLM_BASE_URL = normalize_openai_compatible_base_url(LLM_BASE_URL)

print("LLM base URL:", LLM_BASE_URL)
print("LLM model:", LLM_MODEL_NAME)
print("Test pair-level labels output:", TEST_LABELS_PATH)
print("Test query-level logs output:", TEST_QUERY_LOGS_PATH)

LLM base URL: https://litellm.imt-soft.com/v1
LLM model: local-std-03
Test pair-level labels output: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\test_llm_groundtruth_labels_openai_compatible.jsonl
Test query-level logs output: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\test_llm_groundtruth_query_logs_openai_compatible.jsonl


## 3. Load Shuffled Annotation Items

In [5]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                records.append(json.loads(line))
    return records


def append_jsonl_record(output_path: Path, record: dict) -> None:
    """Append one JSON-serializable record to a JSONL file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("a", encoding="utf-8") as output_file:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()


def compact_text(value: str, max_characters: int) -> str:
    """Clip long text fields so one query prompt remains manageable."""
    text = str(value or "").strip()
    if len(text) > max_characters:
        return text[: max_characters - 3].rstrip() + "..."
    return text




def build_recipe_description_lookup(data_path: Path) -> dict[int, str]:
    """Build doc_id -> recipe description lookup for older annotation files.

    New blinded annotation files already contain recipe_description. This helper
    keeps the labeling notebooks compatible with existing JSONL files that were
    exported before description was included.
    """
    recipe_dataframe = pd.read_csv(data_path).reset_index(drop=True)
    if "doc_id" not in recipe_dataframe.columns:
        recipe_dataframe.insert(0, "doc_id", recipe_dataframe.index.astype(int))
    if "description" not in recipe_dataframe.columns:
        return {}

    return {
        int(row["doc_id"]): compact_text(row.get("description", ""), max_characters=700)
        for _, row in recipe_dataframe[["doc_id", "description"]].iterrows()
    }


def add_missing_recipe_descriptions_to_annotation_items(
    annotation_items: list[dict],
    description_lookup: dict[int, str],
) -> list[dict]:
    """Attach recipe_description when an existing blinded JSONL file does not have it."""
    enriched_items = []
    for annotation_item in annotation_items:
        enriched_item = dict(annotation_item)
        if not str(enriched_item.get("recipe_description", "")).strip():
            doc_id = int(enriched_item["doc_id"])
            enriched_item["recipe_description"] = description_lookup.get(doc_id, "")
        enriched_items.append(enriched_item)
    return enriched_items

def group_annotation_items_by_query(annotation_items: list[dict]) -> list[dict]:
    """Group shuffled annotation items into one prompt payload per query."""
    grouped: dict[int, dict] = {}
    for item in annotation_items:
        query_id = int(item["query_id"])
        grouped.setdefault(
            query_id,
            {
                "query_id": query_id,
                "query_text": str(item["query_text"]),
                "documents": [],
            },
        )
        grouped[query_id]["documents"].append(item)

    query_groups = []
    for query_group in grouped.values():
        query_group["documents"] = sorted(
            query_group["documents"],
            key=lambda item: int(item["blinded_position"]),
        )
        query_groups.append(query_group)
    return sorted(query_groups, key=lambda group: group["query_id"])


def build_candidate_id(blinded_position: int) -> str:
    """Build a short stable candidate ID for the LLM to copy."""
    return f"D{int(blinded_position):03d}"


def build_document_payload(annotation_item: dict) -> dict:
    """Build the full internal document object for one candidate."""
    blinded_position = int(annotation_item["blinded_position"])
    recipe_description = (
        annotation_item.get("recipe_description", "")
        or annotation_item.get("description", "")
    )
    return {
        "candidate_id": build_candidate_id(blinded_position),
        "blinded_position": blinded_position,
        "doc_id": int(annotation_item["doc_id"]),
        "title": compact_text(annotation_item.get("recipe_title", ""), max_characters=180),
        "recipe_type": compact_text(annotation_item.get("recipe_type", ""), max_characters=80),
        "description": compact_text(recipe_description, max_characters=700),
    }


def build_prompt_document(document: dict) -> dict:
    """Build the document object shown to the LLM.

    doc_id is intentionally hidden from the prompt. The LLM only has to copy a short
    candidate_id, and the code maps candidate_id back to doc_id after parsing.
    """
    return {
        "candidate_id": document["candidate_id"],
        "title": document["title"],
        "recipe_type": document["recipe_type"],
        "description": document["description"],
    }


def build_query_payload(query_group: dict) -> dict:
    """Build the full query-level labeling payload."""
    documents = [build_document_payload(item) for item in query_group["documents"]]
    return {
        "query_id": int(query_group["query_id"]),
        "query_text": str(query_group["query_text"]),
        "documents": documents,
        "prompt_documents": [build_prompt_document(document) for document in documents],
    }


def strip_markdown_json_fence(raw_text: str) -> str:
    """Remove common markdown code fences around JSON output."""
    text = str(raw_text).strip()
    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    return fenced_match.group(1).strip() if fenced_match else text


def parse_query_label_response(raw_response_text: str, expected_payload: dict) -> list[dict]:
    """Parse and validate LLM labels for one query using candidate_id.

    The LLM is only expected to copy candidate_id values such as D001, D002, ...
    doc_id is recovered by this parser from the original payload.
    """
    cleaned_response = strip_markdown_json_fence(raw_response_text)
    parsed_response = json.loads(cleaned_response)

    if not isinstance(parsed_response, dict):
        raise ValueError("LLM response must be a JSON object.")
    if int(parsed_response.get("query_id")) != int(expected_payload["query_id"]):
        raise ValueError(
            f"LLM response query_id {parsed_response.get('query_id')} does not match expected {expected_payload['query_id']}."
        )

    labels = parsed_response.get("labels")
    if not isinstance(labels, list):
        raise ValueError("LLM response must contain a labels list.")

    expected_by_candidate_id = {
        str(document["candidate_id"]): document
        for document in expected_payload["documents"]
    }
    expected_candidate_ids = set(expected_by_candidate_id.keys())

    parsed_records = []
    seen_candidate_ids = set()
    unexpected_candidate_ids = []

    for label_item in labels:
        candidate_id = str(label_item.get("candidate_id", "")).strip()
        relevance = int(label_item["relevance"])
        if relevance not in {0, 1, 2, 3}:
            raise ValueError(f"Invalid relevance label {relevance} for candidate_id={candidate_id}.")
        if candidate_id not in expected_by_candidate_id:
            unexpected_candidate_ids.append(candidate_id)
            continue
        if candidate_id in seen_candidate_ids:
            raise ValueError(f"Duplicate label for candidate_id={candidate_id}.")
        seen_candidate_ids.add(candidate_id)

        expected_document = expected_by_candidate_id[candidate_id]
        parsed_records.append(
            {
                "query_id": int(expected_payload["query_id"]),
                "candidate_id": candidate_id,
                "doc_id": int(expected_document["doc_id"]),
                "blinded_position": int(expected_document["blinded_position"]),
                "relevance": relevance,
                "label_match_strategy": "candidate_id",
                "label_reconciled": False,
            }
        )

    if unexpected_candidate_ids:
        raise ValueError(
            "LLM returned unknown candidate_id values: "
            f"{unexpected_candidate_ids[:10]}. Expected IDs include: {sorted(expected_candidate_ids)[:10]}"
        )

    missing_candidate_ids = expected_candidate_ids - seen_candidate_ids
    if missing_candidate_ids:
        raise ValueError(
            f"LLM response missed {len(missing_candidate_ids)} candidates: "
            f"{sorted(missing_candidate_ids)[:10]}"
        )

    return sorted(parsed_records, key=lambda record: record["blinded_position"])


In [6]:
blinded_annotation_items = load_jsonl_records(BLINDED_ANNOTATION_ITEMS_PATH)
recipe_description_lookup = build_recipe_description_lookup(DATA_PATH)
blinded_annotation_items = add_missing_recipe_descriptions_to_annotation_items(
    annotation_items=blinded_annotation_items,
    description_lookup=recipe_description_lookup,
)
query_groups = group_annotation_items_by_query(blinded_annotation_items)

print("Number of blinded annotation items:", len(blinded_annotation_items))
print("Number of query groups:", len(query_groups))
print("Candidates per query:")
print(pd.Series([len(group["documents"]) for group in query_groups]).describe())

query_groups[0]["query_id"], query_groups[0]["query_text"], len(query_groups[0]["documents"])

Number of blinded annotation items: 25000
Number of query groups: 500
Candidates per query:
count    500.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
dtype: float64


(0, 'Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu', 50)

## 4. Prompt and Response Parser

In [7]:
LLM_JUDGE_PROMPT_VERSION = "recipe_relevance_query_batch_title_type_description_v3"

RELEVANCE_JUDGE_SYSTEM_PROMPT = """You are an expert relevance assessor for a Vietnamese recipe search system.
Your task is to label how useful each candidate recipe would be for the user query.
Use only the provided title, recipe_type, and description fields.
Prefer semantic usefulness over exact word matching.
Return valid JSON only. Do not include markdown, comments, or explanations.
"""

RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE = """TASK
Label every candidate document for the given query using a 0-3 graded relevance scale.

IMPORTANT ID RULE
- Each candidate has a short candidate_id such as D001, D002, D003.
- You MUST copy candidate_id values exactly from CANDIDATE DOCUMENTS.
- Do NOT invent candidate_id values.
- Do NOT output doc_id.
- Do NOT output blinded_position.
- Output exactly one label for every candidate_id in CANDIDATE DOCUMENTS.

AVAILABLE EVIDENCE
- Each candidate contains only title, recipe_type, and description.
- Treat title as the strongest signal.
- Use recipe_type and description as supporting signals.
- Do not require evidence outside these three fields.

QUERY
query_id: {query_id}
query_text: {query_text}

CANDIDATE DOCUMENTS
{documents_json}

RELEVANCE DEFINITIONS
3 = Highly relevant:
- The candidate directly satisfies the query intent, or is a natural recipe variant that a user would reasonably accept.
- Exact wording is not required when the title/type/description clearly refer to the same dish, drink, dessert, topping, flavor family, or recipe family.

2 = Relevant:
- The candidate is not the exact requested item, but it is a close substitute or close variant in the same recipe family.
- It would still be useful to show for the query because most of the food/drink intent is preserved.

1 = Somewhat relevant:
- The candidate is related only at a broad level, such as the same general category, occasion, flavor direction, or food/drink group.
- It may be useful for exploration, but it is not a close answer to the query.

0 = Not relevant:
- The candidate would not substantially help the user query.
- Generic Vietnamese food words, generic adjectives, or weak surface overlap are not enough.

CALIBRATION GUIDANCE
- If a candidate is a clear member of the same dish/drink/dessert family as the query, avoid labeling it 0.
- If unsure between two adjacent labels, choose the label that best reflects whether the result would be useful to a real recipe-search user.
- Keep the full 0-3 scale meaningful: use 3 for strong matches, 2 for close variants, 1 for loose related items, and 0 for unrelated items.

MANDATORY OUTPUT RULES
- Return JSON only.
- Return exactly this top-level shape:
{{
  "query_id": {query_id},
  "labels": [
    {{"candidate_id": "D001", "relevance": 0}},
    {{"candidate_id": "D002", "relevance": 3}}
  ]
}}
- The labels list length MUST equal the number of candidate documents.
- Every candidate_id from CANDIDATE DOCUMENTS must appear exactly once.
- relevance must be one of: 0, 1, 2, 3.
"""


def build_relevance_judge_prompt_for_query(query_group: dict) -> dict:
    """Build one prompt for one query and all its candidate documents."""
    payload = build_query_payload(query_group)
    documents_json = json.dumps(payload["prompt_documents"], ensure_ascii=False, indent=2)
    user_prompt = RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE.format(
        query_id=payload["query_id"],
        query_text=payload["query_text"],
        documents_json=documents_json,
    )
    return {
        "system_prompt": RELEVANCE_JUDGE_SYSTEM_PROMPT,
        "user_prompt": user_prompt,
        "prompt_version": LLM_JUDGE_PROMPT_VERSION,
        "payload": payload,
    }


In [8]:
# Inspect one query payload before calling the API.
preview_prompt = build_relevance_judge_prompt_for_query(query_groups[0])
print("query_id:", preview_prompt["payload"]["query_id"])
print("query_text:", preview_prompt["payload"]["query_text"])
print("candidate_count:", len(preview_prompt["payload"]["documents"]))
print(preview_prompt["user_prompt"][:3000])

query_id: 0
query_text: Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu
candidate_count: 50
TASK
Label every candidate document for the given query using a 0-3 graded relevance scale.

IMPORTANT ID RULE
- Each candidate has a short candidate_id such as D001, D002, D003.
- You MUST copy candidate_id values exactly from CANDIDATE DOCUMENTS.
- Do NOT invent candidate_id values.
- Do NOT output doc_id.
- Do NOT output blinded_position.
- Output exactly one label for every candidate_id in CANDIDATE DOCUMENTS.

AVAILABLE EVIDENCE
- Each candidate contains only title, recipe_type, and description.
- Treat title as the strongest signal.
- Use recipe_type and description as supporting signals.
- Do not require evidence outside these three fields.

QUERY
query_id: 0
query_text: Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu

CANDIDATE DOCUMENTS
[
  {
    "candidate_id": "D001",
    "title": "Bánh trung thu dẻo chay không dùng lò nướng",
    "recipe_type": "Món bánh",
   

## 5. OpenAI-Compatible LiteLLM Caller

In [9]:
def call_openai_compatible_query_judge(system_prompt: str, user_prompt: str) -> str:
    """Call the LiteLLM OpenAI-compatible endpoint and return raw text for one query group."""
    try:
        from openai import OpenAI
    except ImportError as import_error:
        raise ImportError("OpenAI SDK is not installed. Install it with `pip install openai`.") from import_error

    client = OpenAI(
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        timeout=LLM_REQUEST_TIMEOUT_SECONDS,
    )

    last_error = None
    for attempt_index in range(1, LLM_MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()
        except Exception as request_error:
            last_error = request_error
            if attempt_index >= LLM_MAX_RETRIES:
                break
            sleep_seconds = min(2 ** attempt_index, 30)
            print(
                f"LLM request failed on attempt {attempt_index}/{LLM_MAX_RETRIES}: "
                f"{request_error!r}. Retrying in {sleep_seconds}s."
            )
            time.sleep(sleep_seconds)

    raise RuntimeError(f"LLM request failed after {LLM_MAX_RETRIES} attempts.") from last_error


## 6. Debug Raw LLM Output Before Parsing

This section is intentionally separate from the normal test runner. It sends a small number of query groups to the OpenAI-compatible endpoint, prints the exact query and candidate documents used in the prompt, then prints the raw LLM response before any parsing or validation happens.

Use this section when the parser raises an error such as an unknown `candidate_id`, a missing label, invalid JSON, or an unexpected output schema. The goal is to make the failure observable before the stricter parser stops the pipeline.


In [10]:
DEBUG_NUM_QUERY_GROUPS = 3
DEBUG_DOCS_TO_PRINT_PER_QUERY = 12
DEBUG_PRINT_FULL_USER_PROMPT = False


def extract_json_object_from_raw_response(raw_response_text: str) -> dict | None:
    """Best-effort JSON extraction used only for debugging malformed model output."""
    cleaned_response_text = strip_markdown_json_fence(raw_response_text)
    try:
        return json.loads(cleaned_response_text)
    except json.JSONDecodeError:
        pass

    object_start_index = cleaned_response_text.find("{")
    object_end_index = cleaned_response_text.rfind("}")
    if object_start_index == -1 or object_end_index == -1 or object_end_index <= object_start_index:
        return None

    try:
        return json.loads(cleaned_response_text[object_start_index : object_end_index + 1])
    except json.JSONDecodeError:
        return None


def display_debug_query_group_inputs(prompt: dict, docs_to_print_per_query: int | None) -> None:
    """Print the query and candidate documents that will be sent to the LLM."""
    payload = prompt["payload"]
    print("=" * 120)
    print(f"query_id: {payload['query_id']}")
    print(f"query_text: {payload['query_text']}")
    print(f"candidate_count: {len(payload['documents'])}")
    print(f"prompt_version: {prompt['prompt_version']}")

    internal_documents_dataframe = pd.DataFrame(payload["documents"])
    internal_columns_to_show = [
        column_name
        for column_name in [
            "candidate_id",
            "blinded_position",
            "doc_id",
            "title",
            "recipe_type",
            "description",
        ]
        if column_name in internal_documents_dataframe.columns
    ]
    print("\nInternal candidate mapping used by the parser:")
    display(internal_documents_dataframe[internal_columns_to_show].head(docs_to_print_per_query))

    prompt_documents_dataframe = pd.DataFrame(payload["prompt_documents"])
    prompt_columns_to_show = [
        column_name
        for column_name in [
            "candidate_id",
            "title",
            "recipe_type",
            "description",
        ]
        if column_name in prompt_documents_dataframe.columns
    ]
    print("\nCandidate documents sent to the LLM:")
    display(prompt_documents_dataframe[prompt_columns_to_show].head(docs_to_print_per_query))

    if DEBUG_PRINT_FULL_USER_PROMPT:
        print("\nFull user prompt:")
        print(prompt["user_prompt"])


def debug_query_group_llm_outputs(
    query_groups: list[dict],
    num_query_groups: int = DEBUG_NUM_QUERY_GROUPS,
    docs_to_print_per_query: int | None = DEBUG_DOCS_TO_PRINT_PER_QUERY,
    parse_after_response: bool = True,
) -> None:
    """Call the LLM query by query and print raw output before strict parsing."""
    selected_query_groups = query_groups[:num_query_groups]
    print(f"Debug query groups selected: {len(selected_query_groups)}")

    for loop_index, query_group in enumerate(selected_query_groups, start=1):
        prompt = build_relevance_judge_prompt_for_query(query_group)
        payload = prompt["payload"]
        expected_candidate_ids = [document["candidate_id"] for document in payload["documents"]]

        print(f"\nDEBUG CALL {loop_index}/{len(selected_query_groups)}")
        display_debug_query_group_inputs(prompt, docs_to_print_per_query)

        raw_response_text = call_openai_compatible_query_judge(
            prompt["system_prompt"],
            prompt["user_prompt"],
        )

        print("\nRaw LLM response before parsing:")
        print(raw_response_text)

        debug_json_object = extract_json_object_from_raw_response(raw_response_text)
        if debug_json_object is None:
            print("\nDebug JSON extraction: failed to parse any JSON object from the raw response.")
        else:
            print("\nDebug JSON extraction: parsed top-level keys:", list(debug_json_object.keys()))
            debug_labels = debug_json_object.get("labels", [])
            if isinstance(debug_labels, list):
                debug_labels_dataframe = pd.DataFrame(debug_labels)
                print(f"Debug JSON extraction: label rows returned by LLM = {len(debug_labels_dataframe)}")
                display(debug_labels_dataframe.head(docs_to_print_per_query))

        if not parse_after_response:
            continue

        try:
            parsed_label_records = parse_query_label_response(raw_response_text, payload)
        except Exception as parse_error:
            print("\nStrict parser error:")
            print(repr(parse_error))
            print("\nExpected candidate_id values:")
            print(expected_candidate_ids)
            continue

        parsed_labels_dataframe = pd.DataFrame(parsed_label_records)
        print("\nStrict parser succeeded. Parsed labels:")
        display(parsed_labels_dataframe.head(docs_to_print_per_query))


### Run the Raw-Output Debug Loop

Run this cell before the normal 20-query test whenever you want to inspect what the LLM actually returned. It will call the endpoint, so only run it after `.env` is configured.


In [11]:
# Run this first when debugging parser failures.
debug_query_group_llm_outputs(
    query_groups=query_groups,
    num_query_groups=DEBUG_NUM_QUERY_GROUPS,
    docs_to_print_per_query=DEBUG_DOCS_TO_PRINT_PER_QUERY,
    parse_after_response=True,
)


Debug query groups selected: 3

DEBUG CALL 1/3
query_id: 0
query_text: Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu
candidate_count: 50
prompt_version: recipe_relevance_query_batch_title_type_description_v3

Internal candidate mapping used by the parser:


,candidate_id,blinded_position,doc_id,title,recipe_type,description
0,D001,1,3056,Bánh trung thu dẻo chay không dùng lò nướng,Món bánh,"Những chiếc bánh trung thu dẻo mềm, ngọt bùi t..."
1,D002,2,2621,Bánh trung thu đậu xanh trứng muối sầu riêng v...,Món bánh,Bánh trung thu đậu xanh sầu riêng có vỏ bánh v...
2,D003,3,2683,Cách chiên bánh bao bằng nồi chiên không dầu v...,Món bánh,"Thay vì phải chiên ngập dầu, bạn hoàn toàn có ..."
3,D004,4,3154,Bánh trung thu 12 con giáp ngộ nghĩnh và vô cù...,Món bánh,Một mâm bánh trung thu thơm ngon thôi vẫn chưa...
4,D005,5,2810,Bánh muffin chuối socola bằng nồi chiên không ...,Món bánh,Bánh muffin chuối socola mềm xốp là món bánh m...
5,D006,6,1946,Bánh trung thu tinh than tre nhân đậu xanh bằn...,Món bánh,Bánh trung thu tinh than tre có sắc đen đẹp mắ...
6,D007,7,2889,Bánh dẻo trung thu khoai lang tím thơm ngon kh...,Món bánh,Bánh dẻo trung thu khoai lang tím màu sắc bắt ...
7,D008,8,1987,Bánh trứng cuộn bằng nồi chiên không dầu giòn ...,Món bánh,Bánh trứng cuộn là món bánh cực kỳ lý tưởng để...
8,D009,9,3050,Bánh trung thu dẻo nhân đậu xanh truyền thống ...,Món bánh,Bánh trung thu dẻo nhân đậu xanh có phần vỏ dẻ...
9,D010,10,1864,"Bánh dày đậu xanh bằng bột nếp, chuẩn vị xứ Quảng",Món bánh,"Bánh dày đậu xanh xứ Quảng dẻo mềm, bùi thơm ă..."



Candidate documents sent to the LLM:


,candidate_id,title,recipe_type,description
0,D001,Bánh trung thu dẻo chay không dùng lò nướng,Món bánh,"Những chiếc bánh trung thu dẻo mềm, ngọt bùi t..."
1,D002,Bánh trung thu đậu xanh trứng muối sầu riêng v...,Món bánh,Bánh trung thu đậu xanh sầu riêng có vỏ bánh v...
2,D003,Cách chiên bánh bao bằng nồi chiên không dầu v...,Món bánh,"Thay vì phải chiên ngập dầu, bạn hoàn toàn có ..."
3,D004,Bánh trung thu 12 con giáp ngộ nghĩnh và vô cù...,Món bánh,Một mâm bánh trung thu thơm ngon thôi vẫn chưa...
4,D005,Bánh muffin chuối socola bằng nồi chiên không ...,Món bánh,Bánh muffin chuối socola mềm xốp là món bánh m...
5,D006,Bánh trung thu tinh than tre nhân đậu xanh bằn...,Món bánh,Bánh trung thu tinh than tre có sắc đen đẹp mắ...
6,D007,Bánh dẻo trung thu khoai lang tím thơm ngon kh...,Món bánh,Bánh dẻo trung thu khoai lang tím màu sắc bắt ...
7,D008,Bánh trứng cuộn bằng nồi chiên không dầu giòn ...,Món bánh,Bánh trứng cuộn là món bánh cực kỳ lý tưởng để...
8,D009,Bánh trung thu dẻo nhân đậu xanh truyền thống ...,Món bánh,Bánh trung thu dẻo nhân đậu xanh có phần vỏ dẻ...
9,D010,"Bánh dày đậu xanh bằng bột nếp, chuẩn vị xứ Quảng",Món bánh,"Bánh dày đậu xanh xứ Quảng dẻo mềm, bùi thơm ă..."



Raw LLM response before parsing:
{
  "query_id": 0,
  "labels": [
    {"candidate_id": "D001", "relevance": 1},
    {"candidate_id": "D002", "relevance": 2},
    {"candidate_id": "D003", "relevance": 0},
    {"candidate_id": "D004", "relevance": 1},
    {"candidate_id": "D005", "relevance": 0},
    {"candidate_id": "D006", "relevance": 2},
    {"candidate_id": "D007", "relevance": 1},
    {"candidate_id": "D008", "relevance": 0},
    {"candidate_id": "D009", "relevance": 1},
    {"candidate_id": "D010", "relevance": 0},
    {"candidate_id": "D011", "relevance": 1},
    {"candidate_id": "D012", "relevance": 1},
    {"candidate_id": "D013", "relevance": 0},
    {"candidate_id": "D014", "relevance": 0},
    {"candidate_id": "D015", "relevance": 2},
    {"candidate_id": "D016", "relevance": 1},
    {"candidate_id": "D017", "relevance": 0},
    {"candidate_id": "D018", "relevance": 3},
    {"candidate_id": "D019", "relevance": 0},
    {"candidate_id": "D020", "relevance": 0},
    {"candida

...

,candidate_id,title,recipe_type,description
0,D001,"Cá ngừ kho tỏi ớt đậm đà, bắt vị cho bữa cơm",Món kho,Cá ngừ kho tỏi ớt là một trong những món kho t...
1,D002,"Mắm cá lóc thơm ngon, đậm vị đơn giản tại nhà",Món khô - mắm,Mắm cá lóc là một trong những đặc sản của miền...
2,D003,Bún cá ngừ hộp ngon nhanh đơn giản cho tuần gi...,Món nước,Nấu ăn là cách tuyệt vời giúp giảm căng thẳng ...
3,D004,Cá ngừ nướng giấy bạc bằng nồi chiên không dầu...,Món nướng,Cá ngừ nướng giấy bạc là một món nướng yêu thí...
4,D005,Cá nhồng 1 nắng chuẩn ngon bằng bộ dao kéo chy...,Món khô - mắm,"Cá nhồng ngoài làm món chiên, kho, nướng thì c..."
5,D006,Sashimi cá ngừ tươi sống đúng chuẩn nhà hàng Nhật,Món gỏi - salad,Bạn mê mẩn món sashimi cá ngừ tươi ngon chuẩn ...
6,D007,3 công thức làm salad cá ngừ cực đơn giản giàu...,Món gỏi - salad,Món salad cá ngừ ở các nhà hàng đồ ăn Tây hấp ...
7,D008,"Ruốc (chà bông) cá chép ngon, giòn, bắt cơm tạ...",Ăn vặt,Ruốc là một món ăn được nhiều người ưa thích b...
8,D009,"Trứng cá ngừ kho tộ đậm đà, cực hao cơm, đơn g...",Món kho,Trứng cá ngừ sữa kho tộ là một trong những món...
9,D010,Bánh canh cá ngừ đậm đà lạ miệng cho cả nhà cù...,Món nước,Cá ngừ là thực phẩm chứa nhiều chất dinh dưỡng...



Raw LLM response before parsing:
{
  "query_id": 2,
  "labels": [
    {"candidate_id": "D001", "relevance": 0},
    {"candidate_id": "D002", "relevance": 0},
    {"candidate_id": "D003", "relevance": 0},
    {"candidate_id": "D004", "relevance": 0},
    {"candidate_id": "D005", "relevance": 0},
    {"candidate_id": "D006", "relevance": 0},
    {"candidate_id": "D007", "relevance": 0},
    {"candidate_id": "D008", "relevance": 2},
    {"candidate_id": "D009", "relevance": 0},
    {"candidate_id": "D010", "relevance": 0},
    {"candidate_id": "D011", "relevance": 0},
    {"candidate_id": "D012", "relevance": 0},
    {"candidate_id": "D013", "relevance": 0},
    {"candidate_id": "D014", "relevance": 0},
    {"candidate_id": "D015", "relevance": 0},
    {"candidate_id": "D016", "relevance": 0},
    {"candidate_id": "D017", "relevance": 0},
    {"candidate_id": "D018", "relevance": 0},
    {"candidate_id": "D019", "relevance": 3},
    {"candidate_id": "D020", "relevance": 0},
    {"candida

,candidate_id,relevance
0,D001,0
1,D002,0
2,D003,0
3,D004,0
4,D005,0
5,D006,0
6,D007,0
7,D008,2
8,D009,0
9,D010,0



Strict parser succeeded. Parsed labels:


,query_id,candidate_id,doc_id,blinded_position,relevance,label_match_strategy,label_reconciled
0,2,D001,6123,1,0,candidate_id,False
1,2,D002,10084,2,0,candidate_id,False
2,2,D003,4286,3,0,candidate_id,False
3,2,D004,4717,4,0,candidate_id,False
4,2,D005,10078,5,0,candidate_id,False
5,2,D006,8201,6,0,candidate_id,False
6,2,D007,8421,7,0,candidate_id,False
7,2,D008,8854,8,2,candidate_id,False
8,2,D009,6310,9,0,candidate_id,False
9,2,D010,4270,10,0,candidate_id,False


## 7. Run 20-Query Test


In [12]:
def load_completed_query_ids(labels_path: Path) -> set[int]:
    """Return query IDs that already have at least one saved label record."""
    if not labels_path.exists():
        return set()
    completed_records = load_jsonl_records(labels_path)
    return {int(record["query_id"]) for record in completed_records}

In [13]:
def judge_query_groups_with_openai(
    query_groups: list[dict],
    labels_output_path: Path,
    query_logs_output_path: Path,
    max_query_groups: int,
    resume: bool,
) -> pd.DataFrame:
    """Small OpenAI-only test runner with one call per query group."""
    selected_query_groups = query_groups[:max_query_groups]
    if not resume:
        for output_path in [labels_output_path, query_logs_output_path]:
            if output_path.exists():
                output_path.unlink()

    completed_query_ids = load_completed_query_ids(labels_output_path) if resume else set()
    saved_label_records = []

    print(f"Test query groups selected: {len(selected_query_groups)}")
    for index, query_group in enumerate(selected_query_groups, start=1):
        query_id = int(query_group["query_id"])
        if query_id in completed_query_ids:
            continue

        prompt = build_relevance_judge_prompt_for_query(query_group)
        raw_response_text = call_openai_compatible_query_judge(prompt["system_prompt"], prompt["user_prompt"])
        try:
            parsed_label_records = parse_query_label_response(raw_response_text, prompt["payload"])
        except Exception as parse_error:
            print("\nStrict parser failed inside the normal runner.")
            display_debug_query_group_inputs(prompt, DEBUG_DOCS_TO_PRINT_PER_QUERY)
            print("\nRaw LLM response before parsing:")
            print(raw_response_text)
            print("\nStrict parser error:")
            print(repr(parse_error))
            raise

        query_log_record = {
            "query_id": query_id,
            "judge_type": "llm",
            "llm_provider": "openai_compatible_litellm",
            "llm_model_name": LLM_MODEL_NAME,
            "prompt_version": prompt["prompt_version"],
            "candidate_count": len(prompt["payload"]["documents"]),
            "parsed_label_count": len(parsed_label_records),
            "raw_response_text": raw_response_text,
        }
        append_jsonl_record(query_logs_output_path, query_log_record)

        for label_record in parsed_label_records:
            output_record = {
                **label_record,
                "judge_type": "llm",
                "llm_provider": "openai_compatible_litellm",
                "llm_model_name": LLM_MODEL_NAME,
                "prompt_version": prompt["prompt_version"],
            }
            append_jsonl_record(labels_output_path, output_record)
            saved_label_records.append(output_record)

        print(
            f"[{index}/{len(selected_query_groups)}] query_id={query_id} "
            f"saved {len(parsed_label_records)} labels"
        )

    return pd.DataFrame(saved_label_records)

In [14]:
# Run this small test after setting LLM_BASE_URL and LLM_MODEL_NAME in .env.
test_label_dataframe = judge_query_groups_with_openai(
    query_groups=query_groups,
    labels_output_path=TEST_LABELS_PATH,
    query_logs_output_path=TEST_QUERY_LOGS_PATH,
    max_query_groups=TEST_NUM_QUERY_GROUPS,
    resume=TEST_RESUME_LABELING,
)
test_label_dataframe.head()

Test query groups selected: 20
[1/20] query_id=0 saved 50 labels
[2/20] query_id=1 saved 50 labels
[3/20] query_id=2 saved 50 labels
[4/20] query_id=3 saved 50 labels
[5/20] query_id=4 saved 50 labels
[6/20] query_id=5 saved 50 labels
[7/20] query_id=6 saved 50 labels
[8/20] query_id=7 saved 50 labels
[9/20] query_id=8 saved 50 labels
[10/20] query_id=9 saved 50 labels
[11/20] query_id=10 saved 50 labels
[12/20] query_id=11 saved 50 labels
[13/20] query_id=12 saved 50 labels
[14/20] query_id=13 saved 50 labels
[15/20] query_id=14 saved 50 labels
[16/20] query_id=15 saved 50 labels
[17/20] query_id=16 saved 50 labels
[18/20] query_id=17 saved 50 labels
[19/20] query_id=18 saved 50 labels
[20/20] query_id=19 saved 50 labels


,query_id,candidate_id,doc_id,blinded_position,relevance,label_match_strategy,label_reconciled,judge_type,llm_provider,llm_model_name,prompt_version
0,0,D001,3056,1,1,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
1,0,D002,2621,2,2,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
2,0,D003,2683,3,0,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
3,0,D004,3154,4,1,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
4,0,D005,2810,5,0,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
